In [1]:
import json

with open("linkedin_scrape_results_20260320_120417.json", 'r') as f:
    result = json.load(f)

with open("catagories.json", "r") as f:
    categories = json.load(f)

In [2]:
import json
from openai import OpenAI
import os 
from dotenv import load_dotenv

load_dotenv()

def recommend_categories(
    profile: dict,
    categories: list[dict] | None = None,
) -> list[dict]:

    api_key = os.getenv("OPENAI_API_KEY")
    client = OpenAI(api_key=api_key)

    if categories is None:
        with open("catagories.json", "r") as f:
            categories = json.load(f)

    system_prompt = """You are an expert at matching professional profiles to relevant report categories.
You will receive a person's LinkedIn profile data and a list of category groups.
From ALL the groups, pick only the subcategories that are relevant to this person.
Return ONLY valid JSON."""

    user_prompt = f"""Analyze this LinkedIn profile and recommend the most relevant subcategories.

## Profile
{json.dumps(profile, indent=2)}

## Available Category Groups
{json.dumps(categories, indent=2)}

Return a JSON object with this exact structure:
{{
    "recommended": [
        {{"category": "<subcategory name>", "relevance_score": <float 0-1>, "reasoning": "<one sentence>"}},
        {{"category": "<subcategory name>", "relevance_score": <float 0-1>, "reasoning": "<one sentence>"}}
    ]
}}

Rules:
- Only pick subcategories from the provided lists (e.g. "Management", "Energy", "Global", "Deep Tech").
- Only include subcategories that are genuinely relevant to this person's profile.
- Return them ranked by relevance_score (highest first)."""

    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_object"},
        temperature=0.3,
    )

    return json.loads(response.choices[0].message.content)["recommended"]

In [3]:
profile = result['results'][0]

url = profile['profile_url']
rec  = recommend_categories(profile, categories= None)
result = {
        url: rec
    }


In [4]:
result

{'http://www.linkedin.com/in/morgan-dammann-a48667140': [{'category': 'management',
   'relevance_score': 0.97,
   'reasoning': 'The profile is centered on a current consultant role at a management consulting firm specializing in business strategy, organizational change, program management, and transformation.'},
  {'category': 'employment',
   'relevance_score': 0.74,
   'reasoning': 'Past experience includes redefining job roles, supporting an engineering rotation program, and customer success and project ownership work that connects to workforce and employment topics.'},
  {'category': 'work-of-future',
   'relevance_score': 0.68,
   'reasoning': 'Experience in organizational design, business transformation, analytics, and role redesign aligns with future-of-work themes around how companies structure teams and capabilities.'},
  {'category': 'america',
   'relevance_score': 0.61,
   'reasoning': "The person's education and work history are based in the United States, including Minne

In [5]:
# print(recommend_categ[0]['http://www.linkedin.com/in/morgan-dammann-a48667140'])

# filter relevance_score': 0.74 > 0.70
filtered_recommend_categ = []
for i in result[url]:
    if i['relevance_score'] > 0.70:
        filtered_recommend_categ.append(i)
filtered_recommend_categ


[{'category': 'management',
  'relevance_score': 0.97,
  'reasoning': 'The profile is centered on a current consultant role at a management consulting firm specializing in business strategy, organizational change, program management, and transformation.'},
 {'category': 'employment',
  'relevance_score': 0.74,
  'reasoning': 'Past experience includes redefining job roles, supporting an engineering rotation program, and customer success and project ownership work that connects to workforce and employment topics.'}]

In [6]:
catagory_slugs = []
for cat in filtered_recommend_categ:
    catagory_slugs.append(cat['category'])
    

In [7]:
catagory_slugs

['management', 'employment']

In [ ]:
"""
subcategory_reports.py

Standalone script with two utility functions:

1. get_report_ids_by_subcategory_slugs(slugs)
   - Validates each slug exists in the subcategories data.
   - Returns a list of dicts with report id and slug belonging to those subcategories.

2. get_categories_with_subcategories()
   - Returns for-you, sector, theme, geography categories (id, name, slug)
     with their subcategories (id, name, slug).

Data source: test_data.json (same directory as this script).
"""

import os
import sys
import json
import logging
from typing import List, Dict, Any

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
)
logger = logging.getLogger("subcategory_reports")

ALLOWED_CATEGORY_SLUGS = {"for-you", "sector", "theme", "region"}

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_FILE = os.path.join(SCRIPT_DIR, "test_data.json")


# ── Load data from JSON ─────────────────────────────────────────────────────

def _load_data() -> Dict[str, Any]:
    logger.debug("Loading data from %s", DATA_FILE)
    with open(DATA_FILE, "r") as f:
        data = json.load(f)
    logger.info(
        "Loaded %d categories, %d subcategories, %d reports, %d report-subcategory links",
        len(data.get("categories", [])),
        len(data.get("subcategories", [])),
        len(data.get("reports", [])),
        len(data.get("report_subcategory", [])),
    )
    return data


_DATA = _load_data()

_CATEGORIES: List[Dict] = _DATA["categories"]
_SUBCATEGORIES: List[Dict] = _DATA["subcategories"]
_REPORTS: List[Dict] = _DATA["reports"]
_REPORT_SUBCATEGORY: List[Dict] = _DATA["report_subcategory"]

_SUBCAT_BY_SLUG: Dict[str, Dict] = {sc["slug"]: sc for sc in _SUBCATEGORIES}
_CAT_BY_ID: Dict[int, Dict] = {cat["id"]: cat for cat in _CATEGORIES}
_REPORT_BY_ID: Dict[int, Dict] = {r["id"]: r for r in _REPORTS}


# ── Function 1 ──────────────────────────────────────────────────────────────

def get_report_ids_by_subcategory_slugs(slugs: List[str]) -> List[Dict[str, Any]]:
    """
    Validate every slug in *slugs* against the subcategories data.
    If any slug is invalid, raises ValueError listing all bad slugs.
    Returns a deduplicated list of dicts {"id": ..., "slug": ...} sorted by id.
    """
    logger.info("Validating %d subcategory slug(s): %s", len(slugs), slugs)

    allowed_cat_ids = {
        cat["id"] for cat in _CATEGORIES
        if cat["slug"] in ALLOWED_CATEGORY_SLUGS
    }

    found = []
    invalid = []
    for slug in slugs:
        sc = _SUBCAT_BY_SLUG.get(slug)
        if sc and sc["cat_id"] in allowed_cat_ids:
            found.append(sc)
        else:
            invalid.append(slug)

    if invalid:
        logger.error("Invalid subcategory slugs found: %s", invalid)
        raise ValueError(f"Invalid subcategory slugs: {invalid}")

    subcat_ids = {sc["id"] for sc in found}
    logger.info("All slugs valid. Matched subcategory IDs: %s", sorted(subcat_ids))

    report_ids = sorted({
        link["report_id"]
        for link in _REPORT_SUBCATEGORY
        if link["subcat_id"] in subcat_ids
    })

    reports = [
        {"id": rid, "slug": _REPORT_BY_ID[rid]["slug"]}
        for rid in report_ids
        if rid in _REPORT_BY_ID
    ]
    logger.info("Found %d report(s) for the given subcategories", len(reports))
    return reports


# ── Function 2 ──────────────────────────────────────────────────────────────

def get_categories_with_subcategories() -> List[Dict[str, Any]]:
    """
    Return for-you, sector, theme, geography categories with their subcategories.

    Returns a list of dicts:
        [
            {
                "id": 1, "name": "Sector", "slug": "sector",
                "subcategories": [
                    {"id": 10, "name": "Healthcare", "slug": "healthcare"},
                    ...
                ]
            },
            ...
        ]
    """
    logger.info("Fetching categories with subcategories (filtered: %s)", ALLOWED_CATEGORY_SLUGS)

    result = []
    for cat in _CATEGORIES:
        if cat["slug"] not in ALLOWED_CATEGORY_SLUGS:
            continue
        subcats = [
            {"id": sc["id"], "name": sc["name"], "slug": sc["slug"]}
            for sc in _SUBCATEGORIES
            if sc["cat_id"] == cat["id"]
        ]
        logger.debug(
            "Category '%s' (id=%d) has %d subcategory(ies)",
            cat["name"], cat["id"], len(subcats),
        )
        result.append({
            "id": cat["id"],
            "name": cat["name"],
            "slug": cat["slug"],
            "subcategories": subcats,
        })

    logger.info("Found %d category(ies)", len(result))
    return result


2026-03-23 16:35:03,452 [INFO] subcategory_reports - Loaded 4 categories, 38 subcategories, 4157 report-subcategory links


In [11]:
report_ids = get_report_ids_by_subcategory_slugs(catagory_slugs)

2026-03-23 16:35:07,486 [INFO] subcategory_reports - Validating 2 subcategory slug(s): ['management', 'employment']
2026-03-23 16:35:07,487 [INFO] subcategory_reports - All slugs valid. Matched subcategory IDs: [3, 20]
2026-03-23 16:35:07,488 [INFO] subcategory_reports - Found 339 report(s) for the given subcategories


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import OpenSearchVectorSearch
from requests_aws4auth import AWS4Auth

/home/dixit/Desktop/report-recommendation/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
BEDROCK_ACCESS_KEY_ID = os.getenv('BEDROCK_ACCESS_KEY_ID')
BEDROCK_SECRET_ACCESS_KEY = os.getenv('BEDROCK_SECRET_ACCESS_KEY')
BEDROCK_REGION_NAME = os.getenv('BEDROCK_REGION_NAME')
BEDROCK_LLM_ID = os.getenv('BEDROCK_LLM_ID')

In [15]:
import boto3
from botocore.config import Config

bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    aws_access_key_id=BEDROCK_ACCESS_KEY_ID,
    aws_secret_access_key=BEDROCK_SECRET_ACCESS_KEY,
    region_name=BEDROCK_REGION_NAME,
    config=Config(read_timeout=3600)
)

In [16]:
# HuggingFace Constants
HUGGINGFACE_EMBEDDINGS_MODEL = os.getenv('HUGGINGFACE_EMBEDDINGS_MODEL')
EMBEDDING_MODEL = HuggingFaceEmbeddings(
    model_name=HUGGINGFACE_EMBEDDINGS_MODEL,
    show_progress=False
)

# AWS OpenSearch Constants
AWS_OPENSEARCH_ACCESS_KEY_ID = os.getenv('AWS_OPENSEARCH_ACCESS_KEY_ID')
AWS_OPENSEARCH_SECRET_ACCESS_KEY = os.getenv('AWS_OPENSEARCH_SECRET_ACCESS_KEY')
AWS_OPENSEARCH_REGION_NAME = os.getenv('AWS_OPENSEARCH_REGION_NAME')
OPENSEARCH_URL = os.getenv('OPENSEARCH_URL')
OPENSEARCH_INDEX = "ghost_research_report_overviews"

service = "es"  # must set the service as 'es'
credentials = boto3.Session(
    aws_access_key_id=AWS_OPENSEARCH_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_OPENSEARCH_SECRET_ACCESS_KEY
).get_credentials()

awsauth = AWS4Auth(
    AWS_OPENSEARCH_ACCESS_KEY_ID,
    AWS_OPENSEARCH_SECRET_ACCESS_KEY,
    AWS_OPENSEARCH_REGION_NAME,
    service,
    session_token=credentials.token       
)

from opensearchpy import RequestsHttpConnection

VECTOR_DB = OpenSearchVectorSearch(
    embedding_function=EMBEDDING_MODEL,
    opensearch_url=OPENSEARCH_URL,
    http_auth=awsauth,
    timeout=300,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    index_name=OPENSEARCH_INDEX,
    engine="faiss"
)

2026-03-23 16:35:24,421 [INFO] sentence_transformers.SentenceTransformer - Use pytorch device_name: cpu
2026-03-23 16:35:24,422 [INFO] sentence_transformers.SentenceTransformer - Load pretrained SentenceTransformer: sentence-transformers/LaBSE
2026-03-23 16:35:25,316 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/sentence-transformers/LaBSE/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-23 16:35:25,317 [WARNING] huggingface_hub.utils._http - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-23 16:35:25,379 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/LaBSE/836121a0533e5664b21c7aacc5d22951f2b8b25b/modules.json "HTTP/1.1 200 OK"
2026-03-23 16:35:25,751 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/sentence-transformers/LaBSE/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary

In [27]:
# ── Step A: LLM-generated search query ───────────────────────────────────────
# Instead of querying OpenSearch with a bare category name like "management",
# we ask GPT to write a single profile-specific search query that covers ALL
# the user's relevant categories at once (one GPT call, not one per category).

def generate_search_query(profile: dict, categories: list[dict]) -> str:
    """Use GPT to craft a single targeted search query from a profile + all relevant categories."""
    api_key = os.getenv("OPENAI_API_KEY")
    client = OpenAI(api_key=api_key)

    categories_block = "\n".join(
        f"- {cat['category']} (relevance: {cat['relevance_score']}): {cat['reasoning']}"
        for cat in categories
    )

    prompt = f"""Based on this LinkedIn profile and their relevant categories,
write a 2-3 sentence description of what kind of research reports this person
would find most valuable. Cover ALL the categories listed below in a single
cohesive description.

Focus on their specific role, industry, company specialties, and strategic interests.
Be specific and concrete — avoid generic descriptions.

Profile:
{json.dumps(profile, indent=2)}

Relevant categories:
{categories_block}

Write ONLY the 2-3 sentence description, nothing else."""

    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )

    return response.choices[0].message.content.strip()

In [28]:
# ── Step B: Single search with the combined profile query ─────────────────────
# One GPT call produces a query covering all categories, then one OpenSearch search.

# Reload profile data (original `result` var was overwritten in cell 2)
with open("linkedin_scrape_results_20260320_120417.json", 'r') as f:
    profiles_data = json.load(f)

profile = profiles_data['results'][0]

search_query = generate_search_query(profile, filtered_recommend_categ)
print(f"Generated query:\n{search_query}\n")

docs = VECTOR_DB.similarity_search_with_score(
    query=search_query,
    k=20,
    filter={"terms": {"metadata.report_id": report_ids}}
)

all_results = []
for doc, sim_score in docs:
    all_results.append({
        "report_id": doc.metadata['report_id'],
        "content": doc.page_content,
        "similarity_score": sim_score,
    })

print(f"Total results: {len(all_results)}")

2026-03-23 16:50:23,090 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Generated query:
As a consultant at Pioneer Management Consulting, Morgan would get the most value from research reports on how mid-market organizations are executing business transformation through organizational change, program management, operating model redesign, and data-driven strategy—especially practical benchmarks tied to cost optimization, restructuring, M&A integration, and organizational effectiveness. Reports that also connect these initiatives to employment outcomes, such as workforce planning, role redesign, talent deployment, rotation or upskilling programs, and change impacts on customer-facing teams, would be particularly relevant given Morgan’s background in job-role definition, project ownership, and customer success.



2026-03-23 16:50:32,398 [INFO] opensearch - POST https://search-consultai-production-zs6diszzmes4wblq2ix5oq5tea.eu-central-1.es.amazonaws.com:443/ghost_research_report_overviews/_search [status:200 request:8.782s]


Total results: 20


In [30]:
all_results

[{'report_id': 142,
  'content': 'This report analyzes the evolving landscape of employment in the banking sector, focusing on the impact of Environmental, Social, and Governance (ESG) factors, digitization, and human capital strategies. It highlights a shift from traditional roles to technology-focused positions, partially driven by rising ESG initiatives and advancements in digital banking. The report delves into global employment trends, future skills requirements, diversity and inclusion efforts, and remote work models. It also presents strategic insights for investors and management to navigate workforce transformations effectively.',
  'similarity_score': 0.55513966},
 {'report_id': 323,
  'content': 'This report delves into how AI and digital disruptions are reshaping the consulting industry. It explores changes in business models, service delivery, and client expectations, driven by AI, cloud computing, and automation. Workforce transformations are also covered, emphasizing AI 

In [31]:
# ── Step C: Re-rank and display ───────────────────────────────────────────────
# Results are already sorted by similarity from OpenSearch. Display top 10.

final_recommendations = sorted(
    all_results,
    key=lambda x: x["similarity_score"],
    reverse=True
)

print(f"{'='*80}")
print(f"TOP 10 RECOMMENDED REPORTS")
print(f"{'='*80}\n")

for i, rec in enumerate(final_recommendations[:10], 1):
    print(f"{i}. Report ID: {rec['report_id']}")
    print(f"   Similarity: {rec['similarity_score']:.4f}")
    print(f"   {rec['content'][:200]}...")
    print()

TOP 10 RECOMMENDED REPORTS

1. Report ID: 142
   Similarity: 0.5551
   This report analyzes the evolving landscape of employment in the banking sector, focusing on the impact of Environmental, Social, and Governance (ESG) factors, digitization, and human capital strategi...

2. Report ID: 323
   Similarity: 0.5540
   This report delves into how AI and digital disruptions are reshaping the consulting industry. It explores changes in business models, service delivery, and client expectations, driven by AI, cloud com...

3. Report ID: 183
   Similarity: 0.5519
   The report 'Future of Work in Real Estate' examines how technological innovations, shifting workforce trends, and policy changes are transforming the global real estate sector. It covers the rise of P...

4. Report ID: 322
   Similarity: 0.5476
   This report explores the transformative impact of AI and automation on various industries including technology, telecommunications, retail, energy, and media. It details how these secto

In [ ]:
# ── Step D (Optional): Hybrid Search — Vector + BM25 via Reciprocal Rank Fusion
# FAISS engine doesn't support native hybrid queries, so we run vector search
# and BM25 keyword search separately, then merge with RRF.
# RRF score = sum(1 / (k + rank)) across both result lists, where k=60 is standard.

from opensearchpy import OpenSearch
from urllib.parse import urlparse

parsed_url = urlparse(OPENSEARCH_URL)
os_client = OpenSearch(
    hosts=[{"host": parsed_url.hostname, "port": parsed_url.port or 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300,
)

# Reuse the same combined query for the hybrid search
hybrid_query_text = generate_search_query(profile, filtered_recommend_categ)
print(f"Hybrid search query: {hybrid_query_text}\n")

# ── Vector search (already have this from Step B, but run fresh for comparison)
vector_docs = VECTOR_DB.similarity_search_with_score(
    query=hybrid_query_text,
    k=20,
    filter={"terms": {"metadata.report_id": report_ids}}
)
vector_ranking = {
    doc.metadata["report_id"]: rank
    for rank, (doc, _) in enumerate(vector_docs)
}

# ── BM25 keyword search via opensearchpy
bm25_query = {
    "size": 20,
    "query": {
        "bool": {
            "must": [
                {"match": {"text": hybrid_query_text}},
            ],
            "filter": [
                {"terms": {"metadata.report_id": report_ids}},
            ],
        }
    },
}
bm25_response = os_client.search(index=OPENSEARCH_INDEX, body=bm25_query)
bm25_ranking = {
    hit["_source"]["metadata"]["report_id"]: rank
    for rank, hit in enumerate(bm25_response["hits"]["hits"])
}

# ── Reciprocal Rank Fusion (k=60 is the standard constant)
RRF_K = 60
all_report_ids = set(vector_ranking.keys()) | set(bm25_ranking.keys())

rrf_scores = {}
for rid in all_report_ids:
    score = 0.0
    if rid in vector_ranking:
        score += 1.0 / (RRF_K + vector_ranking[rid])
    if rid in bm25_ranking:
        score += 1.0 / (RRF_K + bm25_ranking[rid])
    rrf_scores[rid] = score

hybrid_ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

# Build a lookup for content from vector results
content_lookup = {
    doc.metadata["report_id"]: doc.page_content
    for doc, _ in vector_docs
}
for hit in bm25_response["hits"]["hits"]:
    rid = hit["_source"]["metadata"]["report_id"]
    if rid not in content_lookup:
        content_lookup[rid] = hit["_source"].get("text", "")

print(f"Vector returned {len(vector_ranking)} reports, BM25 returned {len(bm25_ranking)} reports")
print(f"Combined unique reports: {len(hybrid_ranked)}")
print(f"\n{'='*80}")
print(f"TOP 10 HYBRID RECOMMENDATIONS (RRF)")
print(f"{'='*80}\n")

for i, (rid, rrf_score) in enumerate(hybrid_ranked[:10], 1):
    v_rank = vector_ranking.get(rid, "-")
    b_rank = bm25_ranking.get(rid, "-")
    content = content_lookup.get(rid, "N/A")
    print(f"{i}. Report ID: {rid}")
    print(f"   RRF Score: {rrf_score:.6f} | Vector Rank: {v_rank} | BM25 Rank: {b_rank}")
    print(f"   {content[:200]}...")
    print()